# Voice Engine — speak any text in your voice

```
Step 1  Prepare your recordings          ← this notebook, now
Step 2  Text → speech in your voice      (English + Hindi)   — coming next
Step 3  engine.speak("text", language="hi")
Later   Add more voices
```

Run the cells **from top to bottom**. In Colab, choose *Runtime → Change runtime type → T4 GPU* (needed from Step 2).

> Your recordings are **private**. They are never in GitHub — you bring them in from Google Drive or by uploading.

## Setup — get the code

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.chdir("/content")
    if not Path("Voice_engine").exists():
        subprocess.run(["git", "clone", "https://github.com/8919134556/Voice_engine.git"], check=True)
    os.chdir("/content/Voice_engine")
    subprocess.run(["git", "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")                                   # running locally from notebooks/
sys.path.insert(0, os.getcwd())
print("Project folder:", os.getcwd())

## Step 1a — bring in your recordings

Your WAV files must end up in `dataset/speaker_01/`. Use **one** of the two cells below.

**Option A — Google Drive.** Upload your `speaker_01` folder to Drive first, then set `DRIVE_FOLDER`
to where it is (the default matches `My Drive/dataset/speaker_01`).

In [ ]:
# Option A: copy from Google Drive
DRIVE_FOLDER = "/content/drive/MyDrive/dataset/speaker_01"     # <- change if your folder is elsewhere

import glob, shutil
from google.colab import drive
drive.mount("/content/drive")                                   # always /content/drive, never the project folder
os.makedirs("dataset/speaker_01", exist_ok=True)
for f in glob.glob(f"{DRIVE_FOLDER}/*.wav"):
    shutil.copy(f, "dataset/speaker_01/")
print("WAV files in dataset/speaker_01:", len(glob.glob("dataset/speaker_01/*.wav")))

**Option B — upload directly** (files disappear when the Colab session ends). Skip if you used Option A.

In [ ]:
# Option B: upload from your computer (select all your WAV files)
import glob
from google.colab import files
os.makedirs("dataset/speaker_01", exist_ok=True)
for name in files.upload():
    os.replace(name, f"dataset/speaker_01/{name}")
print("WAV files in dataset/speaker_01:", len(glob.glob("dataset/speaker_01/*.wav")))

## Step 1b — check and clean the recordings

For every file: is it readable, not silent, not clipped, long enough?
Then a cleaned copy is saved to `dataset_clean/speaker_01/`: mono, 22050 Hz, silence trimmed at
start/end, and **the same loudness for every clip**. Your original files are never changed.

In [ ]:
!python scripts/step1_prepare_dataset.py --voice speaker_01

## Step 1c — listen: original vs cleaned

The cleaned version should sound the same — just without the silence at the edges and at an even volume.

In [ ]:
from IPython.display import Audio, display
name = "audio_001.wav"
print("Original:"); display(Audio(f"dataset/speaker_01/{name}"))
print("Cleaned:");  display(Audio(f"dataset_clean/speaker_01/{name}"))

### Step 1 done when
- the report says **Usable recordings: 20 of 20** (or close), and
- the cleaned audio sounds like you.

Next: **Step 2 — text → speech in your voice.**